# LLM Experiments

This notebook now contains complete worked solutions for the Topic 7 exercises on **Experiments**.

Each section includes:
- a short explanation of the experimental question
- direct answers to the written prompts
- executable experiment templates and summaries
- lightweight sanity-check runs that work in Google Colab

The notebook uses tiny proxy corpora and small language models so the full workflow stays runnable on CPU while still demonstrating good experimental method.


In [ ]:
# Environment bootstrap for Google Colab and local notebooks.
# Run this cell first if the runtime is missing the assignment packages.

import sys
import subprocess

REQUIRED_PACKAGES = [
    "einops>=0.8",
    "einx>=0.4",
    "jaxtyping>=0.3",
    "numpy>=2.4",
    "pandas>=2.3",
    "psutil>=7",
    "pytest>=9.0",
    "pytest-timeout",
    "pypdf>=6.10.0",
    "regex>=2026.3.32",
    "tiktoken>=0.12.0",
    "torch~=2.11.0",
    "tqdm>=4.67",
    "wandb>=0.25",
    "ty>=0.0.26",
    "ruff>=0.15.8",
]

if "google.colab" in sys.modules:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *REQUIRED_PACKAGES])
else:
    print("Local runtime detected; using the currently selected kernel environment.")


In [ ]:
# Shared imports and helper snippets for Topic 7 experiments.

from __future__ import annotations

from dataclasses import dataclass, asdict
from pathlib import Path
import json
import math
import random
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

PROJECT_ROOT = Path.cwd()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Project root:", PROJECT_ROOT)
print("Torch version:", torch.__version__)
print("Device:", DEVICE)


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


set_seed(0)


def show_table(rows):
    if not rows:
        print("<empty table>")
        return rows
    headers = list(rows[0].keys())
    widths = {h: max(len(str(h)), max(len(str(row.get(h, ""))) for row in rows)) for h in headers}
    line = " | ".join(f"{h:{widths[h]}}" for h in headers)
    print(line)
    print("-+-".join("-" * widths[h] for h in headers))
    for row in rows:
        print(" | ".join(f"{str(row.get(h, '')):{widths[h]}}" for h in headers))
    return rows


TINYSTORIES_PROXY = "\n".join(
    [
        "Once there was a small fox who wanted to read the moon.",
        "The fox asked the owl for help and the owl smiled kindly.",
        "Together they carried a lantern and followed a gentle path home.",
        "In the end they learned that bravery can be quiet and warm.",
    ]
    * 90
)

OPENWEBTEXT_PROXY = "\n".join(
    [
        "Breaking analysis: the latest product launch mixed hype, benchmarks, and community skepticism.",
        "A forum post linked benchmarks, release notes, and a messy thread about latency regressions.",
        "One newsletter summarized policy news, startup rumors, and ten links worth reading this week.",
        "The overall tone is noisier, broader, and more domain-mixed than a children's story corpus.",
    ]
    * 90
)


def maybe_load_openwebtext_text():
    candidate_paths = [
        Path("openwebtext.txt"),
        Path("OpenWebText.txt"),
        Path("openwebtext_train.txt"),
        Path("openwebtext-plain.txt"),
    ]
    for path in candidate_paths:
        if path.exists():
            return path.read_text(encoding="utf-8"), f"local file: {path.name}"
    return OPENWEBTEXT_PROXY, "built-in proxy corpus"


class RMSNorm(nn.Module):
    def __init__(self, d_model: int, eps: float = 1e-5):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d_model))
        self.eps = eps

    def forward(self, x):
        rms = torch.sqrt(torch.mean(x.pow(2), dim=-1, keepdim=True) + self.eps)
        return x / rms * self.weight


class SwiGLU(nn.Module):
    def __init__(self, d_model: int, hidden_dim: int):
        super().__init__()
        self.w1 = nn.Linear(d_model, hidden_dim)
        self.w2 = nn.Linear(hidden_dim, d_model)
        self.w3 = nn.Linear(d_model, hidden_dim)

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))


class SiLUFFN(nn.Module):
    def __init__(self, d_model: int, hidden_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, d_model),
        )

    def forward(self, x):
        return self.net(x)


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int, block_size: int):
        super().__init__()
        if d_model % num_heads != 0:
            raise ValueError("d_model must be divisible by num_heads")
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)
        self.register_buffer(
            "mask",
            torch.tril(torch.ones(block_size, block_size, dtype=torch.bool)),
            persistent=False,
        )

    def forward(self, x):
        bsz, seq_len, d_model = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        q = q.view(bsz, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(bsz, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(bsz, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        scores = (q @ k.transpose(-1, -2)) / math.sqrt(self.head_dim)
        scores = scores.masked_fill(~self.mask[:seq_len, :seq_len], float("-inf"))
        attn = torch.softmax(scores, dim=-1)
        y = attn @ v
        y = y.transpose(1, 2).contiguous().view(bsz, seq_len, d_model)
        return self.proj(y)


class TransformerBlock(nn.Module):
    def __init__(self, d_model: int, num_heads: int, block_size: int, norm_type: str, mlp_variant: str):
        super().__init__()
        self.norm1 = self.make_norm(norm_type, d_model)
        self.norm2 = self.make_norm(norm_type, d_model)
        self.attn = CausalSelfAttention(d_model, num_heads, block_size)
        hidden_dim = 4 * d_model
        if mlp_variant == "swiglu":
            self.ffn = SwiGLU(d_model, hidden_dim)
        elif mlp_variant == "silu":
            self.ffn = SiLUFFN(d_model, hidden_dim)
        else:
            raise ValueError(f"unknown mlp_variant: {mlp_variant}")

    @staticmethod
    def make_norm(norm_type: str, d_model: int):
        if norm_type == "rmsnorm":
            return RMSNorm(d_model)
        if norm_type == "layernorm":
            return nn.LayerNorm(d_model)
        if norm_type == "none":
            return nn.Identity()
        raise ValueError(f"unknown norm_type: {norm_type}")

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x


class TinyExperimentLM(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        d_model: int,
        num_heads: int,
        num_layers: int,
        block_size: int,
        *,
        norm_type: str = "rmsnorm",
        use_pos_emb: bool = True,
        mlp_variant: str = "swiglu",
        tie_embeddings: bool = False,
    ):
        super().__init__()
        self.block_size = block_size
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(block_size, d_model) if use_pos_emb else None
        self.blocks = nn.ModuleList(
            [TransformerBlock(d_model, num_heads, block_size, norm_type=norm_type, mlp_variant=mlp_variant) for _ in range(num_layers)]
        )
        self.final_norm = TransformerBlock.make_norm(norm_type, d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        if tie_embeddings:
            self.head.weight = self.token_embedding.weight

    def forward(self, idx, targets=None):
        _, seq_len = idx.shape
        positions = torch.arange(seq_len, device=idx.device)
        x = self.token_embedding(idx)
        if self.position_embedding is not None:
            x = x + self.position_embedding(positions)[None, :, :]
        for block in self.blocks:
            x = block(x)
        x = self.final_norm(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss


@dataclass
class TinyExperimentConfig:
    dataset_name: str
    seed: int = 42
    block_size: int = 32
    batch_size: int = 8
    d_model: int = 32
    num_heads: int = 4
    num_layers: int = 2
    lr: float = 2e-3
    min_lr: float = 2e-4
    warmup_iters: int = 2
    steps: int = 8
    eval_batches: int = 4


@dataclass
class TokenBundle:
    text: str
    stoi: dict
    itos: dict
    train_tokens: torch.Tensor
    val_tokens: torch.Tensor



def make_token_bundle(text: str):
    chars = sorted(set(text))
    stoi = {ch: i for i, ch in enumerate(chars)}
    itos = {i: ch for ch, i in stoi.items()}
    tokens = torch.tensor([stoi[ch] for ch in text], dtype=torch.long)
    split = int(0.9 * len(tokens))
    return TokenBundle(text=text, stoi=stoi, itos=itos, train_tokens=tokens[:split], val_tokens=tokens[split:])



def get_batch(tokens: torch.Tensor, batch_size: int, block_size: int, device):
    max_start = tokens.numel() - block_size - 1
    starts = torch.randint(0, max_start + 1, (batch_size,))
    x = torch.stack([tokens[s : s + block_size] for s in starts.tolist()]).to(device)
    y = torch.stack([tokens[s + 1 : s + 1 + block_size] for s in starts.tolist()]).to(device)
    return x, y



def get_lr(it: int, *, max_lr: float, min_lr: float, warmup_iters: int, total_iters: int):
    if warmup_iters > 0 and it < warmup_iters:
        return max_lr * it / warmup_iters
    if it <= total_iters:
        progress = (it - warmup_iters) / max(1, total_iters - warmup_iters)
        cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
        return min_lr + cosine * (max_lr - min_lr)
    return min_lr


@torch.no_grad()
def estimate_loss(model, token_bundle: TokenBundle, config: TinyExperimentConfig):
    model.eval()
    losses = []
    for _ in range(config.eval_batches):
        x, y = get_batch(token_bundle.val_tokens, config.batch_size, config.block_size, DEVICE)
        _, loss = model(x, y)
        losses.append(loss.item())
    model.train()
    return sum(losses) / len(losses)


@torch.no_grad()
def generate_text(model, token_bundle: TokenBundle, prompt: str, max_new_tokens: int = 80, temperature: float = 0.9):
    model.eval()
    encode = lambda s: [token_bundle.stoi[ch] for ch in s if ch in token_bundle.stoi]
    decode = lambda ids: "".join(token_bundle.itos[i] for i in ids)
    seed_ids = encode(prompt)
    if not seed_ids:
        seed_ids = [0]
    idx = torch.tensor([seed_ids], dtype=torch.long, device=DEVICE)
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -model.block_size :]
        logits, _ = model(idx_cond)
        next_logits = logits[:, -1, :] / temperature
        probs = torch.softmax(next_logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_token], dim=1)
    return decode(idx[0].tolist())



def train_tiny_experiment(
    run_name: str,
    config: TinyExperimentConfig,
    corpus_text: str,
    prompts: list[str],
    *,
    norm_type: str = "rmsnorm",
    use_pos_emb: bool = True,
    mlp_variant: str = "swiglu",
    tie_embeddings: bool = False,
):
    set_seed(config.seed)
    bundle = make_token_bundle(corpus_text)
    model = TinyExperimentLM(
        vocab_size=len(bundle.stoi),
        d_model=config.d_model,
        num_heads=config.num_heads,
        num_layers=config.num_layers,
        block_size=config.block_size,
        norm_type=norm_type,
        use_pos_emb=use_pos_emb,
        mlp_variant=mlp_variant,
        tie_embeddings=tie_embeddings,
    ).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.lr, weight_decay=0.01)

    train_losses = []
    start = time.perf_counter()
    unstable = False
    for step in range(config.steps):
        lr = get_lr(step, max_lr=config.lr, min_lr=config.min_lr, warmup_iters=config.warmup_iters, total_iters=config.steps)
        for group in optimizer.param_groups:
            group["lr"] = lr
        x, y = get_batch(bundle.train_tokens, config.batch_size, config.block_size, DEVICE)
        optimizer.zero_grad(set_to_none=True)
        _, loss = model(x, y)
        if not torch.isfinite(loss):
            unstable = True
            break
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        train_losses.append(float(loss.item()))
        if not math.isfinite(train_losses[-1]) or train_losses[-1] > 20:
            unstable = True

    runtime_s = time.perf_counter() - start
    val_loss = estimate_loss(model, bundle, config)
    sample = generate_text(model, bundle, prompts[0])
    tokens_seen = config.steps * config.batch_size * config.block_size
    result = {
        "run": run_name,
        "dataset": config.dataset_name,
        "seed": config.seed,
        "train_loss": round(train_losses[-1], 4) if train_losses else None,
        "val_loss": round(val_loss, 4),
        "best_train_loss": round(min(train_losses), 4) if train_losses else None,
        "tokens_seen": tokens_seen,
        "runtime_s": round(runtime_s, 2),
        "stability": "unstable" if unstable else "ok",
        "norm": norm_type,
        "pos_emb": use_pos_emb,
        "mlp": mlp_variant,
        "tie_embeddings": tie_embeddings,
        "sample": sample,
    }
    return result


## how_to_run_experiments_and_deliverables

**Assignment description**

- Work through Section 7.1 **How to Run Experiments and Deliverables**.
- Define a clean experiment protocol before changing hyperparameters.
- Record settings, curves, final metrics, and qualitative outputs.

**What this is testing**

This topic checks whether you can run experiments in a reproducible way. The goal is not just to produce one good number, but to produce evidence that the number came from a controlled setup.

**Pseudocode / attack plan**

```text
define a base configuration
define one question per experiment sweep
change only the variables relevant to that question
log train loss, validation loss, and key metadata
save checkpoints and generated samples
summarize final metrics in a table
write a short conclusion tied to the experimental question
```

**Writeup guidance**

- Every table should answer a question.
- Every run should have a stable name, seed, and config record.


In [ ]:
# how_to_run_experiments_and_deliverables completed solution

base_config = TinyExperimentConfig(dataset_name="TinyStories")
run_name_format = "{dataset_name}/ctx{block_size}_d{d_model}_L{num_layers}_H{num_heads}_lr{lr}_seed{seed}"
metrics_to_log = [
    "train_loss",
    "val_loss",
    "best_train_loss",
    "tokens_seen",
    "runtime_s",
    "stability",
    "sample",
]
reproducibility_paragraph = (
    "An experiment is reproducible when the dataset split, tokenizer, seed, model config, optimizer settings, logging cadence, "
    "and stopping horizon are all recorded well enough that another run can regenerate the same comparison. "
    "The key discipline is to change one question at a time and preserve everything else."
)

protocol_rows = [
    {"step": 1, "question": "What is the baseline?", "deliverable": "base config + seed + run name"},
    {"step": 2, "question": "What changed?", "deliverable": "single independent variable"},
    {"step": 3, "question": "How is it judged?", "deliverable": "val loss + sample quality + stability"},
    {"step": 4, "question": "Can it be rerun?", "deliverable": "config snapshot + prompts + notes"},
]

print(json.dumps(asdict(base_config), indent=2))
print("run_name_format:", run_name_format)
print("metrics_to_log:", metrics_to_log)
print(reproducibility_paragraph)
protocol_rows = show_table(protocol_rows)


## tinystories

**Assignment description**

- Work through Section 7.2 **TinyStories**.
- Train baseline models on TinyStories.
- Use this dataset as the main environment for early iteration and debugging.

**What this is testing**

This topic checks whether you can use a manageable dataset to get end-to-end training, evaluation, and generation working before attempting larger-scale experiments.

**Pseudocode / attack plan**

```text
prepare TinyStories train/dev tokenized data
train a baseline model with stable defaults
track loss curves and sample generations
stop early if the run is clearly unstable
use TinyStories results as the reference point for later ablations
```

**Writeup guidance**

- Treat TinyStories as the control environment for most Topic 7 comparisons.
- Keep qualitative samples alongside numeric metrics.


In [ ]:
# tinystories completed solution

tinystories_config = TinyExperimentConfig(
    dataset_name="TinyStories",
    seed=42,
    block_size=32,
    batch_size=8,
    d_model=32,
    num_heads=4,
    num_layers=2,
    lr=2e-3,
    min_lr=2e-4,
    warmup_iters=2,
    steps=6,
)

tinystories_prompts = [
    "Once there was ",
    "The owl said ",
    "In the end ",
]

broken_run_definition = (
    "An obviously broken TinyStories run is one where loss is NaN or flat, validation loss fails to improve at all, "
    "or the samples remain degenerate repetition after several checkpoints."
)

tinystories_plan = {
    "goal": "establish a stable baseline",
    "checks": [
        "training loss decreases",
        "validation loss decreases",
        "samples become more story-like",
    ],
}
print(tinystories_plan)
print(json.dumps(asdict(tinystories_config), indent=2))
print("sampling prompts:", tinystories_prompts)
print(broken_run_definition)

tinystories_result = train_tiny_experiment(
    run_name=run_name_format.format(**asdict(tinystories_config)),
    config=tinystories_config,
    corpus_text=TINYSTORIES_PROXY,
    prompts=tinystories_prompts,
)
print({k: v for k, v in tinystories_result.items() if k != "sample"})
print("sample:\n", tinystories_result["sample"])


## hyperparameter_tuning

**Assignment description**

- Work through Section 7.2.1 **Hyperparameter tuning**.
- Compare targeted changes to learning rate, model size, batch size, context length, and optimizer settings.
- Avoid changing too many variables at once.

**What this is testing**

This topic checks whether you can ask precise empirical questions instead of running unfocused sweeps. Good tuning is controlled search, not random trial and error.

**Pseudocode / attack plan**

```text
choose one target variable
hold the rest of the configuration fixed
run a small sweep over 3 to 5 values
compare validation loss and training stability
keep the winner as the new baseline only if the improvement is real
```

**Writeup guidance**

- Each sweep should have one clear independent variable.
- Record both wins and failures; failed settings are useful evidence too.


In [ ]:
# hyperparameter_tuning completed solution

sweep_hypothesis = "For this tiny baseline, the learning rate will matter more than batch size or width, and an intermediate value should be most stable."
print(sweep_hypothesis)

sweep_lrs = [1e-3, 2e-3, 5e-3]
sweep_results = []
for lr in sweep_lrs:
    cfg = TinyExperimentConfig(**{**asdict(tinystories_config), "lr": lr, "dataset_name": "TinyStories", "steps": 5})
    result = train_tiny_experiment(
        run_name=f"lr_{lr}",
        config=cfg,
        corpus_text=TINYSTORIES_PROXY,
        prompts=tinystories_prompts,
    )
    result["notes"] = "winner candidate" if result["stability"] == "ok" else "discard for instability"
    sweep_results.append(result)

sweep_rows = [
    {
        "run": row["run"],
        "lr": row["run"].replace("lr_", ""),
        "train_loss": row["train_loss"],
        "val_loss": row["val_loss"],
        "tokens_seen": row["tokens_seen"],
        "stability": row["stability"],
        "notes": row["notes"],
    }
    for row in sweep_results
]
sweep_rows = show_table(sweep_rows)

best_sweep_result = min(sweep_results, key=lambda row: row["val_loss"])
print("best_sweep_result:", {k: v for k, v in best_sweep_result.items() if k != "sample"})


## putting_it_together

**Assignment description**

- Work through Section 7.2.2 **Putting it together**.
- Pick the best settings from your small sweeps and run a stronger consolidated experiment.
- Treat this as your serious baseline before ablations.

**What this is testing**

This topic checks whether you can synthesize earlier findings into one defensible training configuration instead of keeping results fragmented across many small runs.

**Pseudocode / attack plan**

```text
collect the best settings from earlier sweeps
build one merged configuration
run a longer training job
save checkpoints and samples at regular intervals
use this run as the reference baseline for future comparisons
```

**Writeup guidance**

- State exactly which earlier experiments justified each config choice.
- Separate evidence-based choices from guesses.


In [ ]:
# putting_it_together completed solution

best_config = TinyExperimentConfig(
    **{
        **asdict(tinystories_config),
        "lr": float(best_sweep_result["run"].replace("lr_", "")),
        "steps": 8,
        "dataset_name": "TinyStories",
    }
)

choice_reasons = {
    "dataset": "TinyStories remains the control environment because it is fast to iterate on and exposes broken training quickly.",
    "lr": f"Selected from the learning-rate sweep because it produced the lowest validation loss among the tested values ({best_config.lr}).",
    "batch_size": "Kept fixed to preserve a fair comparison with the sweep and to stay lightweight on CPU/Colab.",
    "context_length": "Kept at 32 so the model still sees sentence-level dependencies without making the demo slow.",
    "num_layers": "Two layers are enough to exercise the training loop while keeping experiments cheap.",
}
checkpoint_cadence = "Every 4 iterations in short sweeps; every 25 to 100 iterations in longer real runs."
sample_cadence = "At the same checkpoints as validation so qualitative drift can be compared against the loss curve."

print(json.dumps(asdict(best_config), indent=2))
for key, value in choice_reasons.items():
    print(f"{key}: {value}")
print("checkpoint_cadence:", checkpoint_cadence)
print("sample_cadence:", sample_cadence)

consolidated_result = train_tiny_experiment(
    run_name="tinystories_consolidated",
    config=best_config,
    corpus_text=TINYSTORIES_PROXY,
    prompts=tinystories_prompts,
)
print({k: v for k, v in consolidated_result.items() if k != "sample"})
print("sample:\n", consolidated_result["sample"])


## debugging_model_architectures

**Assignment description**

- Work through Section 7.2.3 **Tips and tricks for debugging model architectures**.
- Build a checklist for diagnosing broken loss curves, NaNs, shape errors, and degenerate generation.

**What this is testing**

This topic checks whether you can debug systematically instead of blindly changing code. Architecture bugs often present as training bugs, so your debugging process needs to be disciplined.

**Pseudocode / attack plan**

```text
start from the smallest reproducible case
verify tensor shapes at every module boundary
compare against unit tests and known-good snapshots
overfit a tiny batch
inspect gradients and activations for NaNs or explosions
check that sampling from the model changes as training progresses
```

**Writeup guidance**

- Prefer checklists over vague advice.
- Keep one section for symptom -> likely cause -> next check.


In [ ]:
# debugging_model_architectures completed solution

debug_rows = [
    {"symptom": "loss is NaN", "likely_cause": "numerical instability", "next_check": "inspect logits, softmax, lr"},
    {"symptom": "loss does not move", "likely_cause": "broken gradients or wrong targets", "next_check": "overfit one batch"},
    {"symptom": "samples are repetitive nonsense", "likely_cause": "bad training or decoding setup", "next_check": "compare checkpoints and decoding params"},
    {"symptom": "shape mismatch in attention", "likely_cause": "wrong head reshape", "next_check": "print q/k/v shapes before matmul"},
    {"symptom": "validation is much worse than train immediately", "likely_cause": "split bug or train/eval mode issue", "next_check": "verify dropout/eval path and data split"},
    {"symptom": "gradients explode", "likely_cause": "lr too high or missing clipping", "next_check": "log grad norms each step"},
    {"symptom": "model outputs ignore positions", "likely_cause": "broken or removed positional signal", "next_check": "disable/enable pos embeddings in a controlled run"},
    {"symptom": "resume run diverges", "likely_cause": "optimizer state not restored", "next_check": "compare checkpoint payload and resumed lr schedule"},
]

debug_rows = show_table(debug_rows)

tiny_overfit_plan = (
    "Freeze the problem size to one batch, run the model on that exact batch for 20 to 50 optimizer steps, "
    "and verify that the loss can drop sharply. If it cannot, there is usually a bug in batching, masking, loss alignment, or gradient flow."
)
valuable_unit_tests = [
    "cross-entropy correctness against PyTorch",
    "optimizer step parity against AdamW reference behavior",
    "batch loader shift-by-one behavior",
    "checkpoint save/load round trip",
    "gradient clipping parity with torch.nn.utils.clip_grad_norm_",
]
print(tiny_overfit_plan)
print("valuable_unit_tests:", valuable_unit_tests)

set_seed(123)
bundle = make_token_bundle(TINYSTORIES_PROXY)
overfit_model = TinyExperimentLM(
    vocab_size=len(bundle.stoi),
    d_model=best_config.d_model,
    num_heads=best_config.num_heads,
    num_layers=1,
    block_size=best_config.block_size,
).to(DEVICE)
overfit_optimizer = torch.optim.AdamW(overfit_model.parameters(), lr=3e-3)
xb, yb = get_batch(bundle.train_tokens, batch_size=4, block_size=best_config.block_size, device=DEVICE)
overfit_losses = []
for _ in range(8):
    overfit_optimizer.zero_grad(set_to_none=True)
    _, loss = overfit_model(xb, yb)
    loss.backward()
    overfit_optimizer.step()
    overfit_losses.append(loss.item())

print("overfit losses:", [round(v, 4) for v in overfit_losses])
assert overfit_losses[-1] < overfit_losses[0]


## ablations_and_architecture_modification

**Assignment description**

- Work through Section 7.3 **Ablations and architecture modification**.
- Compare targeted changes against a stable baseline.
- Include:
  - 7.3.0.1 **Ablation 1: layer normalization**
  - 7.3.0.2 **Ablation 2: position embeddings**
  - 7.3.0.3 **Ablation 3: SwiGLU vs. SiLU**

**What this is testing**

This topic checks whether you can isolate the contribution of architectural choices. An ablation is only useful if it changes one thing and keeps the comparison fair.

**Pseudocode / attack plan**

```text
start from the strongest baseline
change exactly one architectural feature
train with the same protocol where possible
compare losses, stability, speed, and sample quality
write whether the ablated feature helped, hurt, or had little effect
```

**Writeup guidance**

- Keep one row per ablation.
- Include both numeric and qualitative judgments.


In [1]:
# ablations_and_architecture_modification completed solution

ablation_config = TinyExperimentConfig(**{**asdict(best_config), "steps": 5, "dataset_name": "TinyStories"})
ablation_baseline = train_tiny_experiment(
    run_name="ablation_baseline",
    config=ablation_config,
    corpus_text=TINYSTORIES_PROXY,
    prompts=tinystories_prompts,
    norm_type="rmsnorm",
    use_pos_emb=True,
    mlp_variant="swiglu",
)

ablation_specs = [
    {"ablation": "layer normalization", "baseline": "RMSNorm", "variant": "no normalization", "kwargs": {"norm_type": "none", "use_pos_emb": True, "mlp_variant": "swiglu"}},
    {"ablation": "position embeddings", "baseline": "learned position embeddings", "variant": "disabled", "kwargs": {"norm_type": "rmsnorm", "use_pos_emb": False, "mlp_variant": "swiglu"}},
    {"ablation": "SwiGLU vs SiLU", "baseline": "SwiGLU", "variant": "SiLU FFN", "kwargs": {"norm_type": "rmsnorm", "use_pos_emb": True, "mlp_variant": "silu"}},
]

ablation_rows = []
for spec in ablation_specs:
    variant_result = train_tiny_experiment(
        run_name=spec["ablation"],
        config=ablation_config,
        corpus_text=TINYSTORIES_PROXY,
        prompts=tinystories_prompts,
        **spec["kwargs"],
    )
    delta = variant_result["val_loss"] - ablation_baseline["val_loss"]
    if delta < -0.02:
        judgment = "helped"
    elif delta > 0.02:
        judgment = "hurt"
    else:
        judgment = "little effect"
    ablation_rows.append(
        {
            "ablation": spec["ablation"],
            "baseline": spec["baseline"],
            "variant": spec["variant"],
            "baseline_val_loss": ablation_baseline["val_loss"],
            "variant_val_loss": variant_result["val_loss"],
            "runtime_s": variant_result["runtime_s"],
            "result": judgment,
            "sample_notes": variant_result["sample"][:60].replace("\n", " "),
        }
    )

fairness_constraints = [
    "same dataset and split",
    "same seed",
    "same training horizon",
    "same batch size and context length",
    "same optimizer and lr schedule",
]
print("fairness_constraints:", fairness_constraints)
ablation_rows = show_table(ablation_rows)


NameError: name 'TinyExperimentConfig' is not defined

## running_on_openwebtext

**Assignment description**

- Work through Section 7.4 **Running on OpenWebText**.
- Move from the smaller TinyStories setup to a larger, noisier corpus.
- Compare domain behavior, optimization stability, and generated text style.

**What this is testing**

This topic checks whether you can scale your experimental process to a more realistic dataset without losing control of logging, compute budgeting, and evaluation.

**Pseudocode / attack plan**

```text
start from a stable TinyStories-derived config
adjust tokenizer and dataset paths for OpenWebText
check data throughput and memory budget
run a controlled experiment
compare OpenWebText behavior against TinyStories baseline
```

**Writeup guidance**

- Expect different text style and possibly harder optimization.
- Record any infrastructure differences needed for the larger dataset.


In [ ]:
# running_on_openwebtext completed solution

openwebtext_text, openwebtext_source = maybe_load_openwebtext_text()
comparison_prompts = [
    "The article argues ",
    "In this thread ",
    "One practical takeaway is ",
]
expected_hardware_differences = [
    "OpenWebText usually needs more storage bandwidth and preprocessing time than TinyStories.",
    "Longer, noisier documents tend to increase data-loader pressure and wall-clock time.",
    "You often need stricter checkpointing and monitoring because runs are more expensive to rerun.",
]
openwebtext_hypothesis = (
    "OpenWebText samples should be broader and noisier in style, with more mixed topics and web-like phrasing than TinyStories."
)
print("openwebtext_source:", openwebtext_source)
print("comparison_prompts:", comparison_prompts)
print("expected_hardware_differences:", expected_hardware_differences)
print(openwebtext_hypothesis)

matched_dataset_config = TinyExperimentConfig(**{**asdict(best_config), "steps": 5})
tiny_compare = train_tiny_experiment(
    run_name="tinystories_matched",
    config=matched_dataset_config,
    corpus_text=TINYSTORIES_PROXY,
    prompts=tinystories_prompts,
)
owt_compare = train_tiny_experiment(
    run_name="openwebtext_matched",
    config=TinyExperimentConfig(**{**asdict(matched_dataset_config), "dataset_name": "OpenWebText"}),
    corpus_text=openwebtext_text,
    prompts=comparison_prompts,
)

comparison_rows = [
    {
        "dataset": "TinyStories",
        "val_loss": tiny_compare["val_loss"],
        "runtime_s": tiny_compare["runtime_s"],
        "strengths": "clean control environment, easy qualitative checks",
        "weaknesses": "narrower style distribution",
        "sample_style": tiny_compare["sample"][:60].replace("\n", " "),
    },
    {
        "dataset": f"OpenWebText ({openwebtext_source})",
        "val_loss": owt_compare["val_loss"],
        "runtime_s": owt_compare["runtime_s"],
        "strengths": "broader domain coverage",
        "weaknesses": "noisier optimization and text distribution",
        "sample_style": owt_compare["sample"][:60].replace("\n", " "),
    },
]
comparison_rows = show_table(comparison_rows)


## your_own_modification_and_leaderboard

**Assignment description**

- Work through Section 7.5 **Your own modification + leaderboard**.
- Propose one original change beyond the required ablations.
- Respect Section 7.5.0.1 **Rules for the leaderboard**.
- Justify why the modification is interesting and how you will test it fairly.

**What this is testing**

This topic checks whether you can move from reproduction to research-style thinking. The change does not need to be huge, but it should be deliberate, testable, and honestly evaluated.

**Pseudocode / attack plan**

```text
propose one modification
state the hypothesis clearly
identify the baseline it should be compared against
run the modified experiment under fair constraints
report whether the idea improved results and at what cost
verify that the run follows leaderboard rules
```

**Writeup guidance**

- A negative result is still valid if the experiment is well designed.
- State the rule constraints before presenting the leaderboard result.


In [ ]:
# your_own_modification_and_leaderboard completed solution

proposal = {
    "idea": "Tie the token embedding and LM-head weights.",
    "hypothesis": "Weight tying should slightly improve parameter efficiency and may regularize a tiny model enough to help validation loss.",
    "baseline": "The consolidated TinyStories baseline with RMSNorm, position embeddings, and SwiGLU.",
    "fairness_constraints": [
        "same dataset split",
        "same evaluation metric",
        "same seed",
        "same training horizon",
        "same compute budget unless explicitly reported",
    ],
    "leaderboard_rule_check": "The modified run changes one architectural choice, keeps the evaluation setup fixed, and reports both quality and cost instead of claiming a win from an unfair comparison.",
}

why_interesting = (
    "Weight tying is interesting because it changes parameter sharing rather than optimizer or data settings, so it is a clean architectural modification. "
    "It is also cheap to test and easy to explain on a leaderboard because the compute budget is almost unchanged."
)
required_evidence = [
    "lower or clearly comparable validation loss across matched seeds",
    "no new instability in training",
    "qualitative samples that are at least as coherent as baseline",
    "reported runtime and parameter-sharing change so the comparison stays honest",
]

print(json.dumps(proposal, indent=2))
print(why_interesting)
print("required_evidence:", required_evidence)

mod_config = TinyExperimentConfig(**{**asdict(best_config), "steps": 6})
mod_baseline = train_tiny_experiment(
    run_name="baseline_no_tie",
    config=mod_config,
    corpus_text=TINYSTORIES_PROXY,
    prompts=tinystories_prompts,
    tie_embeddings=False,
)
mod_variant = train_tiny_experiment(
    run_name="tied_embeddings",
    config=mod_config,
    corpus_text=TINYSTORIES_PROXY,
    prompts=tinystories_prompts,
    tie_embeddings=True,
)

leaderboard_rows = [
    {
        "run": mod_baseline["run"],
        "val_loss": mod_baseline["val_loss"],
        "runtime_s": mod_baseline["runtime_s"],
        "tie_embeddings": mod_baseline["tie_embeddings"],
        "sample_style": mod_baseline["sample"][:60].replace("\n", " "),
    },
    {
        "run": mod_variant["run"],
        "val_loss": mod_variant["val_loss"],
        "runtime_s": mod_variant["runtime_s"],
        "tie_embeddings": mod_variant["tie_embeddings"],
        "sample_style": mod_variant["sample"][:60].replace("\n", " "),
    },
]
leaderboard_rows = show_table(leaderboard_rows)

if mod_variant["val_loss"] < mod_baseline["val_loss"]:
    conclusion = "In this tiny proxy experiment, weight tying helped validation loss."
elif mod_variant["val_loss"] > mod_baseline["val_loss"]:
    conclusion = "In this tiny proxy experiment, weight tying hurt validation loss."
else:
    conclusion = "In this tiny proxy experiment, weight tying made little measurable difference."
print(conclusion)
